# SatQuery: remote-sensing adaptation on a free GPU
Select **GPU / T4** in Colab or Kaggle. Free GPU availability and runtime duration are not guaranteed. This notebook trains an adapter; no training or benchmark success is claimed by the repository.

Copy `model-server/finetune` and your legally obtained dataset subset into the notebook runtime. Keep held-out benchmark tests out of training. BigEarthNet.txt is the primary source: https://txt.bigearth.net/ . Use geographically separated train/validation scenes.

Prepare JSONL records with `scene_id`, `source`, `images` (one or two image paths relative to the JSONL), `question`, and a reference `answer`. Export scientific rasters into documented RGB/SAR display composites first; preserve band order, scaling, scene IDs and sensor labels in your preparation manifest. Do not treat raw SAR as natural RGB.

In [ ]:
from pathlib import Path
# Update these paths after uploading the training code and dataset.
TRAINING_DIR = Path('/content/finetune')  # Kaggle: /kaggle/working/finetune
TRAIN = Path('/content/data/train.jsonl')
VALIDATION = Path('/content/data/validation.jsonl')
OUTPUT = Path('/content/satquery-adapter')
assert (TRAINING_DIR / 'train_rs.py').is_file(), 'Upload the finetune directory first.'
assert TRAIN.is_file() and VALIDATION.is_file(), 'Prepare separate train and validation manifests.'


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(TRAINING_DIR / 'requirements.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
print(torch.cuda.get_device_name(0))


## First check the training path
Run a two-step smoke test before spending GPU quota on a full run. The 7B base uses four-bit weights and limited image resolution. If memory is insufficient, reduce `max_pixels` in the script or select a smaller compatible Qwen2.5-VL model after reviewing its license. Do not silently discard one image in paired examples.

In [ ]:
subprocess.run([sys.executable, str(TRAINING_DIR / 'train_rs.py'), '--train', str(TRAIN),
    '--validation', str(VALIDATION), '--output', str(OUTPUT / 'smoke'), '--max-steps', '2'], check=True)


## Train and retain provenance
Choose a manageable, curated subset for your free runtime. Keep adapters and the adaptation manifest in persistent storage before the session ends. Validation loss is not a VQA benchmark score.

In [ ]:
subprocess.run([sys.executable, str(TRAINING_DIR / 'train_rs.py'), '--train', str(TRAIN),
    '--validation', str(VALIDATION), '--output', str(OUTPUT / 'final'), '--epochs', '1'], check=True)


## Evaluate and serve
Evaluate the base model and adapter on the same held-out VRSBench/RSVQA/CDVQA subsets using their official scorers, and record latency and abstention alongside task metrics. Cross-modal evaluation needs genuine registered optical–SAR pairs.

Merge the adapter into its matching base model with PEFT for deployment, or configure compatible LoRA serving. Point the separate model server at your local OpenAI-compatible server using `MODEL_BACKEND=vllm`, `VLLM_BASE_URL`, and `VLLM_MODEL`. A free notebook is an ephemeral development runtime, not a reliable public production endpoint. Keep a downloaded checkpoint and a tested demo runtime ready.